# Spaceship Titanic — End-to-End Binary Classification

**Kaggle Competition:** Predict which passengers were transported to an alternate dimension when the Spaceship Titanic collided with a spacetime anomaly.

**Problem type:** Binary classification — target variable `Transported` (True/False)

**Dataset:** ~8,700 training passengers, ~4,300 test passengers. Features include passenger demographics, cabin assignment, and spending on ship amenities.

## Notebook Structure

1. Exploratory Data Analysis
2. Feature Engineering  
3. Preprocessing Pipeline
4. Modeling & Evaluation
5. Business Hypothesis Testing
6. Kaggle Submission

---
## 1. Exploratory Data Analysis

Both `train.csv` and `test.csv` are loaded upfront to verify column consistency and plan a preprocessing strategy that works for both. Any statistics computed for preprocessing (medians, modes, encodings) will be fit on train only and applied to test — never the reverse.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
target = 'Transported'

train = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test  = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Columns in train but not test: {set(train.columns) - set(test.columns)}')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/spaceship-titanic/train.csv'

### 1.1 Initial Data Inspection

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
train.isna().sum()

### Observations

- Training set has 8,693 rows and 14 columns (13 features + 1 target)
- Test set has 4,277 rows and 13 columns (no `Transported` column, as expected)
- `CryoSleep` and `VIP` are stored as `object` dtype despite being boolean — this is because pandas cannot store NaN in a true boolean column
- Every column except `PassengerId` and `Transported` has missing values, all in the ~2-3% range — manageable, no column is catastrophically missing
- Spending columns (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`) are heavily right-skewed — median is $0 $ for all five, but max values reach $14,000 - $29,000|
- `Cabin` has structure `Deck/Num/Side` (e.g., `B/0/P`) — will parse into three separate features in notebook 02
- `PassengerId` has structure `gggg_pp` where `gggg` is group ID and `pp` is position within group — will engineer `GroupSize` and `IsAlone` features in notebook 02
- Roughly 55% of passengers appear to be traveling alone based on group ID analysis
- Mean spending is ~$225 for RoomService but median is $0 — more than half of 
  passengers spent nothing; distribution is heavily right-skewed due to a small 
  number of high spenders

### 1.2 Target Variable Analysis

The class balance determines which evaluation metrics are appropriate. A severely imbalanced target makes accuracy misleading.

In [ ]:
# Class distribution — counts and proportions
print("Counts:")
print(train[target].value_counts())
print("\nProportions:")
print(train[target].value_counts(normalize=True).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

sns.countplot(x=target, data=train, ax=ax)

ax.set_title('Target Variable Distribution')
ax.set_xlabel('Transported')
ax.set_ylabel('Count')

# Add percentage labels on top of each bar
total = len(train)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    ax.annotate(percentage, (p.get_x() + p.get_width() / 2, p.get_height() + 20),
                ha='center', fontsize=11)

plt.tight_layout()
plt.show()

### Observations

- Classes are nearly balanced: X% Transported vs Y% Not Transported
- Because classes are balanced, **accuracy is a reasonable primary metric**
- No need for class weighting or oversampling techniques

### 1.3 Missing Values Analysis

Missing values are examined for scope, pattern, and whether missingness itself predicts the target. If missingness is predictive, an indicator feature should be engineered before imputing.

In [ ]:
train.isnull().sum()

In [ ]:
(train.isnull().sum() / len(train)) *100

In [ ]:
cols_missing = train.isnull().sum()
pcnt_missing = (train.isnull().sum() / len(train)) * 100

missingness = pd.DataFrame({
    'missing_count': cols_missing,
    'missing_pct': pcnt_missing.round(2)
}).sort_values(by='missing_count', ascending=False)

missingness

In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(train.isnull(), 
            cbar=False, 
            yticklabels=False)
plt.title('Missing Value Patterns Across Training Set')
plt.tight_layout()
plt.show()

In [ ]:
print(train.groupby(train['CryoSleep'].isnull())['Transported'].mean())

In [ ]:
for cols in train.columns:
    print(train.groupby(train[cols].isnull())['Transported'].mean())

### Observations

- All columns except `PassengerId` and `Transported` have missing values,
  all in the 2-3% range — no column is catastrophically missing
- Missing counts are all distinct, meaning missingness is scattered 
  independently across columns rather than entire rows being wiped
- Heatmap confirms no systematic vertical blocks — missingness appears 
  largely random (consistent with MCAR or mild MAR)
- Transport rate for passengers with missing `CryoSleep` (48.8%) vs 
  not missing (50.4%) is nearly identical — missingness carries no 
  meaningful signal for the target, safe to impute without an indicator feature
- Imputation strategy: median for numerical columns, mode for categoricals,
  computed on train only and applied to test

### 1.4 Categorical Features

Each categorical column is explored for value distribution and transport rate signal. `PassengerId` has a `gggg_pp` structure encoding group membership. `Cabin` encodes `Deck/Number/Side`. Both are parsed for feature engineering.

In [ ]:
train['PassengerId'].str.split('_').str[0].value_counts()

In [ ]:
train['PassengerId'].str.split('_').str[1].value_counts()

In [ ]:
train["HomePlanet"].value_counts().plot(kind = 'bar')
plt.title('Count of HomePlanets')
plt.xlabel('HomePlanet')
plt.ylabel('HomePlanet Count')

In [ ]:
train.groupby('HomePlanet')[target].mean().sort_values(ascending=False)

In [ ]:
train.groupby('CryoSleep')[['RoomService', 'FoodCourt', 'Spa']].mean()

In [ ]:
train.groupby('CryoSleep')[target].mean()

### Observations — CryoSleep

- All passengers with CryoSleep=True spent exactly $0 across every 
  spending column — logical constraint, not just a statistical pattern
- This enables logical imputation in notebook 02: missing CryoSleep 
  values can be inferred from spending columns
- CryoSleep is one of the strongest predictors in the dataset:
    - CryoSleep=True  → 81.8% transport rate
    - CryoSleep=False → 32.9% transport rate
    - 50 percentage point gap from a single binary feature
- Zero spending partially explained by CryoSleep — TotalSpend=0 means 
  different things for sleeping vs awake passengers

In [ ]:
train["Cabin"].str.split('/')

In [ ]:
train["Cabin_Deck"] = train["Cabin"].str.split('/').str[0]

In [ ]:
train.groupby('Cabin_Deck')[target].mean().sort_values(ascending=False)

In [ ]:
train.groupby(['HomePlanet', "Cabin_Deck"])[target].mean().sort_values(ascending=False)

In [ ]:
train["Cabin_Num"] = train["Cabin"].str.split('/').str[1]

In [ ]:
train["Cabin_Side"] = train["Cabin"].str.split('/').str[2]

In [ ]:
train.groupby('Cabin_Side')[target].mean().sort_values(ascending=False)

### additional findings for Deck F

In [ ]:
train[train['HomePlanet'] == 'Mars'].groupby('Cabin_Side')[target].mean()

In [ ]:
train[train['HomePlanet'] == 'Mars'].groupby('Cabin_Deck')[target].mean()

### Observations — Cabin

- Cabin parses cleanly into Deck/Num/Side with no unexpected values except 
  Deck T (only 5 passengers — treat as rare category in preprocessing)
- Strong transport rate signal by deck:
    - High transport: B (73%), C (68%)
    - Mid transport: G (52%), A (50%)  
    - Low transport: F (44%), D (43%), E (36%)
- Deck likely correlates with HomePlanet — B/C are Europa passengers, 
  F/G are Earth passengers — will check for multicollinearity later
- Cabin_Side split is nearly 50/50 (P vs S) — will check transport 
  rate by side

In [ ]:
train["Destination"].value_counts().plot(kind = "bar")
plt.title('Count of Destinations')
plt.xlabel('Destinations')
plt.ylabel('Destinations Count')

In [ ]:
train.groupby('Destination')[target].mean().sort_values(ascending=False)

In [ ]:
train["VIP"].value_counts()

In [ ]:
train.groupby('VIP')[target].mean()

In [ ]:
train["Name"].str.split(" ")

In [ ]:
train["FirstName"] = train["Name"].str.split(" ").str[0]

In [ ]:
train["LastName"] = train["Name"].str.split(" ").str[1]

In [ ]:
train["LastName"].value_counts()

In [ ]:
# Do passengers with common surnames cluster by HomePlanet?
common_surnames = train['LastName'].value_counts().head(10).index
train[train['LastName'].isin(common_surnames)].groupby('LastName')['HomePlanet'].value_counts()

### Observations — Categorical Features Summary

- HomePlanet is strongly predictive: Europa (66%) > Mars (52%) > Earth (42%)
- HomePlanet and Cabin_Deck are correlated — Europa clusters in B/C, 
  Earth clusters in F/G — they partially encode the same signal
- VIP has a lower transport rate (38%) than non-VIP (51%) — likely 
  confounded by spending behavior rather than a direct causal relationship
- Cabin_Side: Starboard (55%) vs Port (45%) — consistent 10pt gap
- Key interaction found: Mars/Deck F transports at 65% vs Earth/Deck F 
  at 29% — same deck, very different rates by planet. Tree models will 
  capture this naturally, Logistic Regression will not

### 1.5 Numerical Features

The five spending columns are zero-inflated — median $0 across all of them. Spending is compared before and after `log1p` transformation to understand the true distribution of non-zero values.

In [ ]:
sns.histplot(data = train, x = "Age", kde = True, hue = target, bins = 40)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

### Observations — Age

- Distribution is roughly unimodal with peak around age 25, 
  steadily declining through older ages
- Notable spike at age 0 (infants) with ~67% transport rate — 
  possibly related to higher CryoSleep rates for infants
- Transport rate is approximately 50/50 for ages 10+ meaning 
  raw age has limited linear predictive power for adults
- Will engineer `AgeBin` feature in notebook 02 to capture 
  the child vs adult threshold effect that tree models would 
  find naturally but Logistic Regression would miss
- KDE curves cross at approximately age 12-15, confirming the 
  child threshold effect visually — under 12 skews transported, 
  12+ is approximately 50/50

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train['RoomService'].dropna(), bins=50)
axes[0].set_title('RoomService — Raw')

axes[1].hist(np.log1p(train['RoomService'].dropna()), bins=50)
axes[1].set_title('RoomService — log1p transformed')

plt.tight_layout()
plt.show()

In [ ]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

fig, axes = plt.subplots(len(spend_cols), 2, figsize=(12, 20))

for i, col in enumerate(spend_cols):
    # Left plot — raw
    axes[i, 0].hist(train[col].dropna(), bins=50)
    axes[i, 0].set_title(f'{col} — Raw')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Count')
    
    # Right plot — log1p transformed
    axes[i, 1].hist(np.log1p(train[col].dropna()), bins=50)
    axes[i, 1].set_title(f'{col} — log1p Transformed')
    axes[i, 1].set_xlabel(f'log1p({col})')
    axes[i, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Look at only passengers who actually spent something
spenders = train[train['RoomService'] > 0]['RoomService']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(spenders, bins=50)
axes[0].set_title('RoomService — Non-zero only (Raw)')

axes[1].hist(np.log1p(spenders), bins=50)
axes[1].set_title('RoomService — Non-zero only (log1p)')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(spend_cols), figsize=(18, 5))

for i, col in enumerate(spend_cols):
    sns.boxplot(x=target, y=np.log1p(train[col]), data=train, ax=axes[i])
    axes[i].set_title(f'{col} vs Transported')
    axes[i].set_xlabel('Transported')
    axes[i].set_ylabel(f'log1p({col})')

plt.tight_layout()
plt.show()

### Observations — Spending Columns (Boxplots)

- RoomService, Spa, VRDeck all show higher median spending for 
  Transported=False — these are leisure amenities, active enjoyment 
  of the ship correlates with not being transported
- FoodCourt and ShoppingMall show approximately equal medians 
  across both classes — utility spending is not predictive
- Will engineer LeisureSpend and UtilitySpend separately in 
  notebook 02 to capture this distinction

In [ ]:
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Cabin_Deck', 'Cabin_Side']

for col in cat_cols:
    print(f"--- {col} ---")
    print(train.groupby(col)[target].mean())
    print("\n")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.barplot(x=col, y=target, data=train, 
                ax=axes[i], estimator='mean')
    axes[i].set_title(f'Transport Rate by {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Transport Rate')
    axes[i].set_ylim(0, 1)

plt.tight_layout()
plt.show()

### Observations — Categorical Features vs Transported

- CryoSleep is the dominant categorical predictor — 82% transport 
  rate when True vs 33% when False, a 50 point gap no other feature 
  approaches
- Cabin_Deck shows strong signal: B (73%) and C (68%) are high-transport 
  decks, E (36%) and T (20%) are low — but Deck T has only 5 passengers 
  so its rate is unreliable (visible in wide confidence interval)
- HomePlanet: Europa (66%) > Mars (52%) > Earth (42%) — likely 
  correlated with Cabin_Deck assignment
- Destination: 55 Cancri e passengers transport at 61%, notably 
  higher than TRAPPIST-1e (47%)
- Cabin_Side: Starboard (55%) vs Port (45%) — consistent but modest gap
- VIP transports at lower rate (38%) than non-VIP (50%) — counterintuitive, 
  likely confounded by higher leisure spending among VIPs
- Consistent y-axis (0 to 1) across all plots makes gaps directly 
  comparable — CryoSleep gap is visually ~3x larger than any other feature

In [ ]:
train.info()

In [ ]:
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    # Apply log1p to spending columns but not Age
    x_data = np.log1p(train[col]) if col != 'Age' else train[col]
    
    sns.boxplot(x=target, y=x_data, data=train, ax=axes[i])
    axes[i].set_title(f'Transport Rate by {col}')
    axes[i].set_xlabel('Transported')
    axes[i].set_ylabel(f'log1p({col})' if col != 'Age' else col)  

plt.tight_layout()
plt.show()

### Observations — Numerical Features vs Transported

- Age shows almost no difference between transported and non-transported 
  passengers — median ~27 for both, similar spread. Raw age is not 
  predictive; AgeBin will capture the child threshold effect instead
- RoomService, Spa, VRDeck (leisure): non-transported passengers have 
  noticeably higher median spending — boxes sit clearly above zero for 
  Transported=False, near zero for Transported=True
- FoodCourt, ShoppingMall (utility): both classes show median near zero, 
  consistent with zero-inflation from CryoSleep passengers dominating 
  the Transported=True group
- The Transported=True spending pattern across all five columns is the 
  CryoSleep signature — transported passengers are disproportionately 
  cryosleep passengers with $0 spending across all amenities
- Confirms need for LeisureSpend and UtilitySpend as separate engineered 
  features to isolate genuine spending behavior from CryoSleep zeros

In [ ]:
train_corr = train.copy()
train_corr[target] = train_corr[target].astype(int)
train_corr.corr(numeric_only=True)

In [ ]:
sns.heatmap(train_corr.corr(numeric_only=True), annot = True, fmt = '.2f', cmap = 'coolwarm')

### Observations — Correlation Heatmap

- Strongest linear correlations with Transported:
    - RoomService (-0.24), Spa (-0.22), VRDeck (-0.21) — all negative,
      confirming leisure spending correlates with not being transported
    - FoodCourt (0.05), ShoppingMall (0.01) — near zero, utility 
      spending has no linear relationship with target
- CryoSleep excluded from heatmap (object dtype) — would be the 
  strongest predictor if included, confirmed in Section 5.3
- All correlations are relatively weak (<0.25) suggesting relationships 
  are largely non-linear — tree-based models will capture this better 
  than linear models
- Spending columns show mild inter-correlations (FoodCourt/VRDeck 0.23, 
  FoodCourt/Spa 0.22) — expected since high-spending passengers tend to 
  use multiple amenities

---
## 2. Feature Engineering

All feature engineering is encapsulated in `engineer_features()` — a single function applied identically to train and test. This guarantees both datasets receive identical treatment and prevents the common mistake of forgetting a feature on one split.

**New features:**
- `Group`, `GroupSize`, `IsAlone` — from PassengerId `gggg_pp` structure
- `Cabin_Deck`, `Cabin_Num`, `Cabin_Num_Binned`, `Cabin_Side` — from Cabin `Deck/Num/Side`
- `LastName`, `FamilySize` — from surname frequency
- `AgeBin` — Child / Teen / YoungAdult / Adult / Senior
- `TotalSpend`, `LeisureSpend`, `UtilitySpend`, `IsAwakeZeroSpender` — from spending patterns
- `CryoSleep`, `VIP` — converted to 0/1 integer

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
import joblib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

import joblib

# Local modules
import sys

RANDOM_STATE = 42
target = 'Transported'

In [ ]:
train = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')

In [ ]:
print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

In [ ]:
train["Group"] = train["PassengerId"].str.split("_").str[0]

In [ ]:
train["GroupSize"] = train.groupby("Group")["Group"].transform("count")

In [ ]:
train["IsAlone"] = (train.GroupSize == 1).astype(int)

In [ ]:
train["Cabin_Deck"] = train["Cabin"].str.split('/').str[0]
train["Cabin_Num"] = train["Cabin"].str.split('/').str[1].astype(float)
train["Cabin_Side"] = train["Cabin"].str.split('/').str[2]

In [ ]:
train["Cabin_Num_Binned"] = pd.qcut(train["Cabin_Num"], q = 3, labels = ["Low", "Mid", "High"])

In [ ]:
train.head()

In [ ]:
train["LastName"] = train["Name"].str.split(" ").str[1]

In [ ]:
train["FirstName"] = train["Name"].str.split(" ").str[0]

In [ ]:
train["FamilySize"] = train.groupby("LastName")["LastName"].transform("count")

In [ ]:
train["AgeBin"] = pd.cut(train["Age"], bins = [0, 12, 17, 25, 65, 200],  labels = ['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'], include_lowest = True)

In [ ]:
# 2.5 Spending features
# Fill NaN spending with 0 before computing derived features
# NaN spending most likely means $0 spent
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

for col in spend_cols:
    train[col] = train[col].fillna(0)

In [ ]:
train["TotalSpend"] = train[spend_cols].sum(axis = 1)

In [ ]:
train["LeisureSpend"] = train["RoomService"] + train["Spa"] + train["VRDeck"]

In [ ]:
train["UtilitySpend"] = train["FoodCourt"] + train["ShoppingMall"]

In [ ]:
train["IsAwakeZeroSpender"] = ((train['CryoSleep'] == False) & (train['TotalSpend'] == 0)).astype(int)

In [ ]:
def engineer_features(df):
    """
    Apply all feature engineering to a DataFrame.
    Works identically on both train and test.
    """
    df = df.copy()  # don't modify the original
    
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    
    # 2.1 PassengerId features
    df["Group"] = df["PassengerId"].str.split("_").str[0]
    df["GroupSize"] = df.groupby("Group")["Group"].transform("count")
    df["IsAlone"] = (df.GroupSize == 1).astype(int)
    
    # 2.2 Cabin features  
    df["Cabin_Deck"] = df["Cabin"].str.split('/').str[0]
    df["Cabin_Num"] = df["Cabin"].str.split('/').str[1].astype(float)
    df["Cabin_Side"] = df["Cabin"].str.split('/').str[2]
    df["Cabin_Num_Binned"] = pd.qcut(df["Cabin_Num"], q = 3, labels = ["Low", "Mid", "High"])
    
    # 2.3 Name features
    df["LastName"] = df["Name"].str.split(" ").str[1]
    df["FamilySize"] = df.groupby("LastName")["LastName"].transform("count")
    
    # 2.4 Age features
    df["AgeBin"] = pd.cut(df["Age"], bins = [0, 12, 17, 25, 65, 200],  labels = ['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'], include_lowest = True)
    
    # 2.5 Spending features
    # Fill NaN spending with 0 before deriving features
    for col in spend_cols:
        df[col] = df[col].fillna(0)

    df["TotalSpend"] = df[spend_cols].sum(axis=1)
    df["LeisureSpend"] = df["RoomService"] + df["Spa"] + df["VRDeck"]
    df["UtilitySpend"] = df["FoodCourt"] + df["ShoppingMall"]
    df["IsAwakeZeroSpender"] = ((df['CryoSleep'] == False) & (df['TotalSpend'] == 0)).astype(int)

    return df

In [ ]:
train_raw = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test_raw = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')

train = engineer_features(train_raw)
test = engineer_features(test_raw)

print(f"Train columns: {train.shape[1]}")
print(f"Test columns:  {test.shape[1]}")
print(f"Column difference: {set(train.columns) - set(test.columns)}")

In [ ]:
train[train[spend_cols].sum(axis=1) == 0]

In [ ]:
def logical_impute(df):
    df = df.copy()
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    
    # Rule 1: CryoSleep NaN + zero spending → CryoSleep = 1
    cond_null = df['CryoSleep'].isnull()
    cond_zero_spend = df[spend_cols].sum(axis=1) == 0
    rule1 = cond_null & cond_zero_spend
    df.loc[rule1, 'CryoSleep'] = 1

    # Rule 2: CryoSleep NaN + any spending > 0 → CryoSleep = 0
    cond_null = df['CryoSleep'].isnull()
    cond_spend = df[spend_cols].sum(axis=1) > 0
    rule2 = cond_null & cond_spend
    df.loc[rule2, 'CryoSleep'] = 0

    # Rule 3: CryoSleep = 1 + spending NaN → spending = 0
    cond_cryo_true = (df['CryoSleep'] == 1)
    for col in spend_cols:
        df.loc[cond_cryo_true, col] = df.loc[cond_cryo_true, col].fillna(0)

    return df

In [ ]:
train = logical_impute(train)
test = logical_impute(test)

In [ ]:
print("Missing values after logical imputation:")
print(train.isnull().sum()[train.isnull().sum() > 0])

### Observations — Logical Imputation

- CryoSleep: resolved all 217 missing values using spending rules
  - Passengers with zero spending → imputed CryoSleep = True
  - Passengers with any spending > 0 → imputed CryoSleep = False
- All raw spending columns (RoomService, FoodCourt, etc.): 
  resolved ~110-120 missing values each by filling NaN with 0 
  before computing derived features
- LeisureSpend, UtilitySpend, TotalSpend: 0 missing values 
  after fillna applied inside engineer_features
- Remaining missing values (~179-203 per column) are in 
  demographic and identifier columns with no logical constraints
- These will be handled by SimpleImputer in the Pipeline:
  median for numerical (Age, Cabin_Num), mode for categorical 
  (HomePlanet, Destination, VIP, Cabin_Deck, Cabin_Side, AgeBin)

In [ ]:
train["IsAlone"].value_counts()

In [ ]:
train.groupby("IsAlone")[target].mean()

In [ ]:
train["AgeBin"].value_counts()

In [ ]:
train.groupby("AgeBin")[target].mean()

In [ ]:
# Does group size beyond just IsAlone add signal?
train.groupby('GroupSize')[target].mean().sort_index()

In [ ]:
# Does the leisure vs utility distinction hold in the engineered features?
print(train.groupby(pd.cut(train['LeisureSpend'], 
      bins=[-1, 0, 100, 1000, 100000], 
      labels=['Zero', 'Low', 'Mid', 'High']))[target].mean())

### Observations — Engineered Feature Validation

- `IsAlone`: group travelers transport at 56.7% vs solo at 45.2% — 
  11 point gap confirms feature is predictive. Counterintuitively, 
  group travelers are more likely to be transported
- `AgeBin`: child threshold effect confirmed cleanly — Children 
  transport at 69.9% vs 45-55% for all adult categories. Age only 
  matters as a threshold (child vs adult), not as a continuous value
- `LeisureSpend` binned: near-perfect monotonic relationship with 
  transport rate — Zero (78%) → Low (66%) → Mid (29%) → High (14%). 
  64 point gap between zero and high spenders. Strongest engineered 
  signal in the dataset
- Combined leisure spending is more predictive than any individual 
  spending column alone — confirms feature engineering decision
- Zero leisure spend inflated to 78% by CryoSleep passengers — 
  `IsAwakeZeroSpender` separates this population

In [ ]:
# Columns to drop entirely — raw identifiers
drop_cols = ['PassengerId', 'Name', 'Cabin', 'Group', 'LastName']

# Numerical columns — impute with median, scale
num_cols = [
    # Demographics
    'Age', 'GroupSize', 'FamilySize', 'Cabin_Num',
    # Boolean flags (already 0/1)
    'CryoSleep', 'VIP', 'IsAlone', 'IsAwakeZeroSpender',
    # Raw spending
    'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
    # Engineered spending
    'TotalSpend', 'LeisureSpend', 'UtilitySpend'
]

# Categorical columns — impute with mode, one-hot encode
cat_cols = [
    'HomePlanet', 'Destination',
    'Cabin_Deck', 'Cabin_Side', 'Cabin_Num_Binned',
    'AgeBin'
]

In [ ]:
all_accounted = set(drop_cols + num_cols + cat_cols + [target])
all_columns = set(train.columns)

print(f"Unaccounted columns: {all_columns - all_accounted}")
print(f"Columns in lists but not in train: {all_accounted - all_columns}")

In [ ]:
def cast_to_str(X):
    """Convert all columns to string dtype for OHE compatibility."""
    return pd.DataFrame(X).astype(str)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ("to_str", FunctionTransformer(cast_to_str, feature_names_out='one-to-one')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

---
## 3. Preprocessing Pipeline

A scikit-learn `ColumnTransformer` handles imputation, encoding, and scaling in a single object. The pipeline is fit on `X_train` only. Validation and test sets are transformed using the statistics learned from training — never refit.

**Numerical:** `SimpleImputer(median)` → `StandardScaler`

**Categorical:** `SimpleImputer(mode)` → string cast → `OneHotEncoder(drop='first')`

After preprocessing: **34 features** (16 numerical + 18 from one-hot encoding). Processed arrays are saved to `/kaggle/working/` so the modeling section can load them directly.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
], remainder='drop')

In [ ]:
X = train.drop(columns=[target])
y = train[target]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [ ]:
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"\ny_train transport rate: {y_train.mean():.3f}")
print(f"y_val transport rate:   {y_val.mean():.3f}")

In [ ]:
# Fit ONLY on X_train, then transform all three sets
X_train_processed =  preprocessor.fit_transform(X_train)  # fit_transform
X_val_processed = preprocessor.transform(X_val)    # transform only
X_test_processed = preprocessor.transform(test)   # transform only

In [ ]:
# Check shapes
print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_val_processed shape:   {X_val_processed.shape}")
print(f"X_test_processed shape:  {X_test_processed.shape}")

# Verify no NaN values remain
print(f"\nNaN in X_train: {np.isnan(X_train_processed).sum()}")
print(f"NaN in X_val:   {np.isnan(X_val_processed).sum()}")

# Get OHE category names for categorical columns
ohe = preprocessor.named_transformers_['cat']['encoder']
cat_feature_names = ohe.get_feature_names_out()

# Combine with numerical column names
feature_names = np.array(num_cols + list(cat_feature_names))

print(f"Total features: {len(feature_names)}")
print(feature_names)

In [ ]:
# Build cleaner feature names
ohe = preprocessor.named_transformers_['cat']['encoder']
cat_feature_names = [f"{col}_{val}" for col, cats in 
                     zip(cat_cols, ohe.categories_) 
                     for val in cats[1:]]  # [1:] because drop='first'

feature_names = np.array(num_cols + cat_feature_names)
print(feature_names)

In [ ]:
# Defined at module level so joblib can pickle it
def cast_to_str(x):
    return x.astype(str)

class SpaceshipPreprocessor:
    """
    Reusable preprocessing pipeline for binary classification projects.
    Fits on training data only and transforms all splits consistently.
    """
    
    def __init__(self, num_cols, cat_cols):
        """Store column configuration and initialize pipeline as None."""
        self.num_cols = num_cols
        self.cat_cols = cat_cols
        self.preprocessor = None      # will be set after fit
        self.feature_names_ = None    # will be set after fit
    
    def _build_pipeline(self):
        """Build and return the ColumnTransformer pipeline."""
        
        numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
        ])

        categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('to_str', FunctionTransformer(cast_to_str, feature_names_out='one-to-one')),
        ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
        ])

        return ColumnTransformer(transformers=[
        ('num', numeric_pipeline, self.num_cols),
        ('cat', categorical_pipeline, self.cat_cols)
        ], remainder='drop')
    
    def fit_transform(self, X_train, X_val, X_test):
        """
        Fit pipeline on X_train only.
        Transform X_train, X_val, and X_test.
        Returns tuple: (X_train_p, X_val_p, X_test_p)
        """
        # Build and fit the pipeline
        self.preprocessor = self._build_pipeline()
        
        # fit_transform on train, transform only on val and test
        X_train_p = self.preprocessor.fit_transform(X_train)
        X_val_p = self.preprocessor.transform(X_val) 
        X_test_p = self.preprocessor.transform(X_test)
        
        # Build feature names
        self.feature_names_ = self._get_feature_names()
        
        return X_train_p, X_val_p, X_test_p
    
    def _get_feature_names(self):
        """Build readable feature names after fitting."""
        ohe = self.preprocessor.named_transformers_['cat']['encoder']
        cat_feature_names = [f"{col}_{val}" for col, cats in 
                             zip(self.cat_cols, ohe.categories_) 
                             for val in cats[1:]]
        return np.array(self.num_cols + cat_feature_names)
    
    def summary(self, X_train_p, X_val_p, X_test_p):
        """Print shapes, NaN counts, and feature names."""
        print(f"X_train shape: {X_train_p.shape}")
        print(f"X_val shape:   {X_val_p.shape}")
        print(f"X_test shape:  {X_test_p.shape}")
        print(f"\nNaN in X_train: {np.isnan(X_train_p).sum()}")
        print(f"NaN in X_val:   {np.isnan(X_val_p).sum()}")
        print(f"\nTotal features: {len(self.feature_names_)}")
        print(f"Feature names:\n{self.feature_names_}")

In [ ]:
sp = SpaceshipPreprocessor(num_cols, cat_cols)
X_train_processed, X_val_processed, X_test_processed = sp.fit_transform(X_train, X_val, test)
sp.summary(X_train_processed, X_val_processed, X_test_processed)
feature_names = sp.feature_names_

### Observations

*How many features do we have after preprocessing? How does that compare to the raw column count? Are there any remaining NaN values?*

In [ ]:
# Save preprocessor object
joblib.dump(sp.preprocessor, '/kaggle/working/preprocessor.joblib')  # ← correct

# Save processed arrays
np.save('/kaggle/working/X_train.npy', X_train_processed)
np.save('/kaggle/working/X_val.npy', X_val_processed)
np.save('/kaggle/working/X_test.npy', X_test_processed)
np.save('/kaggle/working/y_train.npy', y_train)
np.save('/kaggle/working/y_val.npy', y_val)

# Save feature names
np.save('/kaggle/working/feature_names.npy', feature_names)

print("All files saved successfully.")

In [ ]:
import os
saved_files = [
    '/kaggle/working/preprocessor.joblib',
    '/kaggle/working/X_train.npy',
    '/kaggle/working/X_val.npy', 
    '/kaggle/working/X_test.npy',
    '/kaggle/working/y_train.npy',
    '/kaggle/working/y_val.npy',
    '/kaggle/working/feature_names.npy'
]
for f in saved_files:
    exists = os.path.exists(f)
    print(f"{'✓' if exists else '✗'} {f}")

## Summary

### Features Engineered
- **PassengerId:** Group, GroupSize, IsAlone (solo traveler flag)
- **Cabin:** Cabin_Deck, Cabin_Num, Cabin_Num_Binned (Low/Mid/High), Cabin_Side
- **Name:** LastName, FamilySize (surname frequency)
- **Age:** AgeBin (Child/Teen/YoungAdult/Adult/Senior)
- **Spending:** TotalSpend, LeisureSpend, UtilitySpend, IsAwakeZeroSpender
- **Boolean conversion:** CryoSleep and VIP converted from object to 0/1 integer

### Preprocessing Pipeline
- **Numerical columns (16):** SimpleImputer(median) → StandardScaler
- **Categorical columns (6):** SimpleImputer(mode) → cast_to_str → OneHotEncoder(drop='first')
- Pipeline fit on X_train only — transform applied to val and test
- Encapsulated in reusable SpaceshipPreprocessor class in src/preprocessor.py

### Data Sizes
- X_train: (6954, 34)
- X_val:   (1739, 34)
- X_test:  (4277, 34)
- Total features after preprocessing: 34 (16 numerical + 18 from OHE)


---
## 4. Modeling & Evaluation

Three classifiers are trained in order of increasing complexity. Logistic Regression establishes the performance floor. The gap between LR and tree models quantifies non-linear structure in the data.

All models are evaluated on a stratified 20% validation hold-out. **AUC-ROC** measures how well the model ranks transported passengers above non-transported ones regardless of threshold — more informative than accuracy for comparing model quality.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                              recall_score, f1_score, confusion_matrix,
                              ConfusionMatrixDisplay)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from bayes_opt import BayesianOptimization
from scipy.stats import chi2_contingency
import shap

In [ ]:
X_train = np.load('/kaggle/working/X_train.npy')
X_val   = np.load('/kaggle/working/X_val.npy')
X_test  = np.load('/kaggle/working/X_test.npy')
y_train = np.load('/kaggle/working/y_train.npy').astype(int)
y_val   = np.load('/kaggle/working/y_val.npy').astype(int)
feature_names = np.load('/kaggle/working/feature_names.npy', allow_pickle=True)

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')
print(f'Features: {len(feature_names)}')

In [ ]:
# XGBoost requires integer labels
y_train = y_train.astype(int)
y_val = y_val.astype(int)

# Verify everything loaded correctly
print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape} | classes: {np.unique(y_train)}")
print(f"y_val:   {y_val.shape}   | classes: {np.unique(y_val)}")
print(f"Features: {len(feature_names)}")

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    """Print full classification metrics for a model."""
    
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"AUC-ROC:   {roc_auc_score(y_true, y_prob):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1:        {f1_score(y_true, y_pred):.4f}")

In [ ]:
# Instantiate
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

# Fit
log_reg.fit(X_train, y_train)

# Predict
y_pred = log_reg.predict(X_val)
y_prob = log_reg.predict_proba(X_val)[:, 1]

# Evaluate
evaluate_model('Logistic Regression', y_val, y_pred, y_prob)

# Cross-validation
cv_scores = cross_val_score(log_reg, X_train, y_train, 
                             cv=StratifiedKFold(5), scoring='accuracy')
print(f"\nCV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

### Observations — Logistic Regression

- **Accuracy: 79.41%** — strong baseline. The data has meaningful linear 
  signal, particularly from binary features like CryoSleep and the 
  spending columns which translate well into linear separability
- **AUC-ROC: 0.8879** — high for a linear model. 88.8% of the time the 
  model correctly ranks a transported passenger above a non-transported 
  one, regardless of threshold
- **Precision (79.23%) vs Recall (80.14%)** — nearly identical, meaning 
  the model makes roughly equal false positives and false negatives. 
  No systematic bias toward either error type
- **F1: 0.7968** — consistent with accuracy on this balanced dataset, 
  as expected when classes are ~50/50
- **CV: 79.26% ± 0.42%** — extremely tight standard deviation. All five 
  folds produced nearly identical results, confirming the val score is 
  trustworthy and not a lucky split
- **CV vs Val gap: 0.15 points** — negligible difference between 
  cross-validated and held-out performance. No signs of overfitting
- This baseline sets the floor — any tree model that can't beat 79.4% 
  is not worth the added complexity

In [ ]:
rfc = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)

# Fit
rfc.fit(X_train, y_train)

# Predict
y_pred = rfc.predict(X_val)
y_prob = rfc.predict_proba(X_val)[:, 1]

# Evaluate
evaluate_model('Random Forest', y_val, y_pred, y_prob)

# Cross-validation
cv_scores = cross_val_score(rfc, X_train, y_train, 
                             cv=StratifiedKFold(5), scoring='accuracy')
print(f"\nCV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


In [ ]:
importances = pd.Series(rfc.feature_importances_, index=feature_names)
importances.nlargest(15).sort_values().plot(kind='barh', figsize=(8, 6))
plt.title('Random Forest — Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

### Observations — Random Forest

- **Accuracy: 80.56%** — 1.15 point improvement over Logistic Regression, 
  confirming non-linear structure exists but the dataset is not 
  dramatically non-linear
- **AUC-ROC: 0.8950** — modest improvement over LR (0.8879)
- **Precision (82.33%) > Recall (78.20%)** — RF is more conservative than 
  LR about predicting Transported=True. Fewer false positives but more 
  missed transported passengers compared to LR
- **CV: 79.78% ± 0.53%** — stable across folds, slightly wider std than 
  LR which is normal for a more complex model
- **CV vs Val gap: 0.78 points** — healthy, no serious overfitting
- **Top feature importances: LeisureSpend, TotalSpend, Cabin_Num** — 
  spending features dominate because RF's impurity-based importance 
  favors continuous features over binary ones. CryoSleep likely 
  underranked here — SHAP will give a fairer picture in Section 8
- **Cabin_Num in top 3** — suggests a finer gradient along the ship's 
  corridor that Low/Mid/High binning was too coarse to fully capture

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    verbosity=0
)

# Fit
xgb.fit(X_train, y_train)

# Predict
y_pred = xgb.predict(X_val)
y_prob = xgb.predict_proba(X_val)[:, 1]

# Evaluate
evaluate_model('XGBoost', y_val, y_pred, y_prob)

# Cross-validation
cv_scores = cross_val_score(xgb, X_train, y_train, 
                             cv=StratifiedKFold(5), scoring='accuracy')
print(f"\nCV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


### Observations — XGBoost (Default Parameters)

- **Accuracy: 81.37%** — 0.81 point improvement over Random Forest, 
  confirming boosting finds patterns that parallel trees miss
- **AUC-ROC: 0.9114** — meaningful jump from RF (0.8950). XGBoost 
  crosses 0.91, ranking transported passengers above non-transported 
  ones significantly more reliably
- **Precision (81.72%) vs Recall (81.16%)** — nearly balanced, unlike 
  RF's precision-heavy profile. Boosting's focus on hard cases naturally 
  produces more balanced error types
- **CV: 80.40% ± 0.36%** — tightest std of all three models despite being 
  the most complex. Stable and trustworthy
- **CV vs Val gap: 0.97 points** — healthy, no overfitting concern
- Default parameters already outperform tuned Random Forest — strong 
  baseline before Bayesian optimization in Section 5

In [ ]:
def xgb_objective(learning_rate, max_depth, subsample,
                  colsample_bytree, n_estimators):
    """Objective for Bayesian Optimization. Returns CV accuracy."""
    model = XGBClassifier(
        learning_rate=learning_rate,
        max_depth=int(max_depth),         # BO returns floats, XGB needs ints
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        n_estimators=int(n_estimators),
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        verbosity=0
    )
    scores = cross_val_score(model, X_train, y_train,
                             cv=StratifiedKFold(3), scoring='accuracy')
    return scores.mean()

In [ ]:
param_bounds = {
    'learning_rate':    (0.01, 0.3),
    'max_depth':        (3, 8),
    'subsample':        (0.6, 1.0),
    'colsample_bytree': (0.6, 1.0),
    'n_estimators':     (100, 500)
}
optimizer = BayesianOptimization(
    f=xgb_objective,
    pbounds=param_bounds,
    random_state=RANDOM_STATE,
    verbose=2
)
optimizer.maximize(init_points=5, n_iter=20)

In [ ]:
best_params = optimizer.max['params']
best_params['max_depth'] = int(best_params['max_depth'])
best_params['n_estimators'] = int(best_params['n_estimators'])

print("Best parameters found:")
for k, v in best_params.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

xgb_tuned = XGBClassifier(
    **best_params,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    verbosity=0
)
xgb_tuned.fit(X_train, y_train)

y_pred_tuned = xgb_tuned.predict(X_val)
y_prob_tuned = xgb_tuned.predict_proba(X_val)[:, 1]

evaluate_model('XGBoost (Tuned)', y_val, y_pred_tuned, y_prob_tuned)

cv_scores_tuned = cross_val_score(xgb_tuned, X_train, y_train,
                                   cv=StratifiedKFold(5), scoring='accuracy')
print(f"\nCV Accuracy: {cv_scores_tuned.mean():.4f} ± {cv_scores_tuned.std():.4f}")

joblib.dump(xgb_tuned, '/kaggle/working/xgb_tuned.joblib')
print("Tuned model saved.")

### Observations — Bayesian Optimization

- **Best params:** learning_rate=0.063, max_depth=4, n_estimators=216,
  subsample=0.810, colsample_bytree=0.773
- Learning rate dropped from 0.1 → 0.063 as predicted — more conservative
  updates generalize better. Max depth stayed at 4 — confirms the dataset
  doesn't require deep trees
- **Optimization range was narrow (0.785–0.808)** — the model is near its
  ceiling with the current feature set. More tuning iterations would not
  help significantly; better features would
- **Accuracy dropped 0.12 points** vs default — within noise range, not
  meaningful
- **AUC-ROC improved** (0.9114 → 0.9128) and **F1 improved** (0.8144 →
  0.8160) — the tuned model is holistically better even if val accuracy
  is marginally lower
- **Key lesson:** optimizing for accuracy shaped a model with slightly
  different precision/recall tradeoff. Optimizing for AUC-ROC directly
  might produce even better ranking quality
- **CV std tightened** (0.0036 → 0.0029) — tuned model is more stable
  across folds
- Tuned model saved to `../models/xgb_tuned.joblib`

In [ ]:
results = pd.DataFrame([
    {'Model': 'Logistic Regression', 'CV Acc': 0.7926, 'Val Acc': 0.7941, 'AUC-ROC': 0.8879, 'F1': 0.7968},
    {'Model': 'Random Forest',       'CV Acc': 0.7978, 'Val Acc': 0.8056, 'AUC-ROC': 0.8950, 'F1': 0.8021},
    {'Model': 'XGBoost (default)',   'CV Acc': 0.8040, 'Val Acc': 0.8137, 'AUC-ROC': 0.9114, 'F1': 0.8144},
    {'Model': 'XGBoost (tuned)',     'CV Acc': 0.8041, 'Val Acc': 0.8125, 'AUC-ROC': 0.9128, 'F1': 0.8160},
]).set_index('Model')

print(results.round(4))

In [ ]:
results.plot(kind='bar', figsize=(12, 5))
plt.title('Model Comparison — Validation Set')
plt.ylabel('Score')
plt.xticks(rotation=15)
plt.ylim(0.7, 1.0)  # zoom into the relevant range
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Observations — Model Comparison

- Consistent improvement across all models: LR → RF → XGBoost
- AUC-ROC improves most dramatically (0.888 → 0.913) — models get 
  significantly better at ranking even when accuracy gains are modest
- XGBoost default vs tuned difference is nearly invisible on the chart — 
  tuning found marginal gains, confirming the model is near its ceiling 
  with the current feature set
- CV and Val accuracy track closely for all models — no overfitting detected
- The modest LR → tree model gap (~2 points) suggests the problem has 
  meaningful but not dominant non-linear structure
- Best overall model: XGBoost tuned — highest AUC-ROC and F1 even though 
  val accuracy is marginally below XGBoost default

In [ ]:
models_list = [
    ('Logistic Regression', log_reg),
    ('Random Forest', rfc),
    ('XGBoost Default', xgb),
    ('XGBoost Tuned', xgb_tuned)
]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, model) in zip(axes, models_list):
    cm = confusion_matrix(y_val, model.predict(X_val))
    ConfusionMatrixDisplay(cm, display_labels=['Not Transported', 'Transported']).plot(ax=ax)
    ax.set_title(name)
plt.tight_layout()
plt.show()

### Observations — Confusion Matrix Analysis

| Model | FP | FN | Total Errors |
|-------|----|----|-------------|
| Logistic Regression | 184 | 174 | 358 |
| Random Forest | 147 | 191 | 338 |
| XGBoost Default | 159 | 165 | 324 |
| XGBoost Tuned | 173 | 153 | 326 |

- **Logistic Regression:** most balanced error distribution (184 FP vs 
  174 FN) — no systematic bias toward either error type
- **Random Forest:** precision-heavy — fewest false positives (147) but 
  most missed transported passengers (191 FN). Conservative about 
  predicting Transported=True
- **XGBoost Default:** most balanced of the tree models (159 FP vs 165 FN) 
  and fewest total errors (324)
- **XGBoost Tuned:** recall-heavy — fewest false negatives (153) and most 
  correct transported predictions (723 TP), but more false alarms (173 FP)
- Model choice depends on error cost: if missing a transported passenger 
  is costly → XGBoost Tuned. If false alarms are costly → Random Forest
- For Kaggle (symmetric error cost), XGBoost Default and Tuned are 
  essentially equivalent

In [ ]:
# Build explainer and compute SHAP values
explainer = shap.TreeExplainer(xgb_tuned)
shap_values = explainer.shap_values(X_val)

# Summary plot — global importance with direction
shap.summary_plot(shap_values, X_val, feature_names=feature_names)

# Bar plot — mean absolute SHAP ranking
shap.summary_plot(shap_values, X_val, 
                  feature_names=feature_names, plot_type='bar')

# Waterfall — explain one specific passenger
shap.plots.waterfall(explainer(X_val)[0])

### Observations — SHAP Explainability

- **Top features by mean absolute SHAP:**
    1. LeisureSpend
    2. HomePlanet_Europa  
    3. CryoSleep
- **LeisureSpend ranks above CryoSleep** — not a surprise on reflection. 
  LeisureSpend is continuous (wider SHAP value range) and partially 
  encodes CryoSleep (CryoSleep passengers always have LeisureSpend=0). 
  SHAP distributes credit between correlated features rather than 
  assigning it all to one
- **HomePlanet_Europa in second place** — stronger than EDA suggested. 
  Likely capturing both direct planet effects and indirect Cabin_Deck 
  effects since Europa passengers cluster in high-transport decks B and C
- **CryoSleep third** — still highly decisive directionally. In the 
  summary dot plot, CryoSleep=0 (blue) clusters hard left and 
  CryoSleep=1 (red) clusters hard right — the cleanest directional 
  split of any feature
- SHAP gives a fairer picture than Random Forest feature importance 
  which ranked spending features first for the wrong reason 
  (continuous features inflate impurity-based importance)
- Prediction from EDA: CryoSleep would be top feature — partially 
  confirmed, it's top 3 and most directionally decisive even if not 
  ranked first by mean absolute SHAP

In [ ]:
train_raw = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
ct = pd.crosstab(train_raw['HomePlanet'], train_raw['Transported'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f"Chi-squared statistic: {chi2:.4f}")
print(f"p-value: {p:.8f}")
print(f"Degrees of freedom: {dof}")

In [ ]:
train_raw['Cabin_Side'] = train_raw['Cabin'].str.split('/').str[2]
ct = pd.crosstab(train_raw['Cabin_Side'], train_raw['Transported'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f"Chi-squared statistic: {chi2:.4f}")
print(f"p-value: {p:.8f}")
print(f"Degrees of freedom: {dof}")

In [ ]:
train_raw['Group'] = train_raw['PassengerId'].str.split('_').str[0]
train_raw['GroupSize'] = train_raw.groupby('Group')['Group'].transform('count')
train_raw['IsAlone'] = (train_raw['GroupSize'] == 1)

ct = pd.crosstab(train_raw['IsAlone'], train_raw['Transported'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f"Chi-squared statistic: {chi2:.4f}")
print(f"p-value: {p:.8f}")
print(f"Degrees of freedom: {dof}")

### Observations — Business Hypothesis Testing

| Hypothesis | Chi-squared | p-value | Significant? |
|---|---|---|---|
| HomePlanet affects transport rate | 324.90 | <0.0001 | ✓ Yes |
| Cabin Side affects transport rate | 91.06 | <0.0001 | ✓ Yes |
| Group travel affects transport rate | 112.10 | <0.0001 | ✓ Yes |

**Hypothesis 1 — HomePlanet:**
The strongest result by far (chi-squared=324.90). The 24-point transport 
rate gap between Europa (66%) and Earth (42%) is not sampling noise — 
HomePlanet is a genuine predictor of transportation. This is consistent 
with SHAP ranking HomePlanet_Europa as the second most important feature.

**Hypothesis 2 — Cabin Side:**
Despite only a 10-point gap (Starboard 55% vs Port 45%), the result is 
highly significant (chi-squared=91.06). Side of the ship genuinely affects 
transport probability — likely reflecting physical proximity to the anomaly.

**Hypothesis 3 — Group Travel:**
Significant despite the smallest raw gap (57% vs 45%, 11 points). The 
chi-squared of 112.10 exceeds Cabin Side despite the smaller effect — 
purely because both groups are large and the signal is consistent. 
Traveling in a group genuinely increases transport probability.

**Key lesson:** Statistical significance and practical significance are 
different things. All three patterns are real — none are sampling noise. 
But HomePlanet's 24-point gap has more practical impact than Cabin Side's 
10-point gap, even though both are technically "significant."

In [ ]:
predictions = xgb_tuned.predict(X_test).astype(bool)  # Kaggle expects True/False

In [ ]:
test_raw = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')
passenger_ids = test_raw['PassengerId']

In [ ]:
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Transported': predictions
})
sample = pd.read_csv('/kaggle/input/spaceship-titanic/sample_submission.csv')
print(f"Sample format:\n{sample.head()}")
print(f"\nOur submission:\n{submission.head()}")
print(f"\nShape: {submission.shape}")  # should be (4277, 2)
print(f"Dtype: {submission['Transported'].dtype}")  # should be bool

In [ ]:
submission.to_csv('/kaggle/working/submission.csv', index=False)

---
## 5. Business Hypothesis Testing

Statistical hypothesis testing answers whether EDA patterns are real or could be random sampling noise. The chi-squared test asks: if this variable had no effect on transport rate, how likely is it we'd see a difference this large by chance? A p-value below 0.05 means less than 5% chance — strong evidence the pattern is real.

Three patterns from EDA are tested. Hypothesis testing is used here for domain understanding, not model selection — those decisions are driven by the metrics in Section 4.

In [ ]:
train_raw = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
ct = pd.crosstab(train_raw['HomePlanet'], train_raw['Transported'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f"Chi-squared statistic: {chi2:.4f}")
print(f"p-value: {p:.8f}")
print(f"Degrees of freedom: {dof}")

In [ ]:
train_raw['Cabin_Side'] = train_raw['Cabin'].str.split('/').str[2]
ct = pd.crosstab(train_raw['Cabin_Side'], train_raw['Transported'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f"Chi-squared statistic: {chi2:.4f}")
print(f"p-value: {p:.8f}")
print(f"Degrees of freedom: {dof}")

In [ ]:
train_raw['Group'] = train_raw['PassengerId'].str.split('_').str[0]
train_raw['GroupSize'] = train_raw.groupby('Group')['Group'].transform('count')
train_raw['IsAlone'] = (train_raw['GroupSize'] == 1)

ct = pd.crosstab(train_raw['IsAlone'], train_raw['Transported'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f"Chi-squared statistic: {chi2:.4f}")
print(f"p-value: {p:.8f}")
print(f"Degrees of freedom: {dof}")

### Observations — Business Hypothesis Testing

| Hypothesis | Chi-squared | p-value | Significant? |
|---|---|---|---|
| HomePlanet affects transport rate | 324.90 | <0.0001 | ✓ Yes |
| Cabin Side affects transport rate | 91.06 | <0.0001 | ✓ Yes |
| Group travel affects transport rate | 112.10 | <0.0001 | ✓ Yes |

**Hypothesis 1 — HomePlanet:**
The strongest result by far (chi-squared=324.90). The 24-point transport 
rate gap between Europa (66%) and Earth (42%) is not sampling noise — 
HomePlanet is a genuine predictor of transportation. This is consistent 
with SHAP ranking HomePlanet_Europa as the second most important feature.

**Hypothesis 2 — Cabin Side:**
Despite only a 10-point gap (Starboard 55% vs Port 45%), the result is 
highly significant (chi-squared=91.06). Side of the ship genuinely affects 
transport probability — likely reflecting physical proximity to the anomaly.

**Hypothesis 3 — Group Travel:**
Significant despite the smallest raw gap (57% vs 45%, 11 points). The 
chi-squared of 112.10 exceeds Cabin Side despite the smaller effect — 
purely because both groups are large and the signal is consistent. 
Traveling in a group genuinely increases transport probability.

**Key lesson:** Statistical significance and practical significance are 
different things. All three patterns are real — none are sampling noise. 
But HomePlanet's 24-point gap has more practical impact than Cabin Side's 
10-point gap, even though both are technically "significant."

---
## 6. Kaggle Submission

The tuned XGBoost model generates predictions on the held-out test set. Kaggle expects boolean True/False predictions in a two-column CSV.

In [ ]:
predictions = xgb_tuned.predict(X_test).astype(bool)  # Kaggle expects True/False

In [ ]:
test_raw = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')
passenger_ids = test_raw['PassengerId']

In [ ]:
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Transported': predictions
})
sample = pd.read_csv('/kaggle/input/spaceship-titanic/sample_submission.csv')
print(f"Sample format:\n{sample.head()}")
print(f"\nOur submission:\n{submission.head()}")
print(f"\nShape: {submission.shape}")  # should be (4277, 2)
print(f"Dtype: {submission['Transported'].dtype}")  # should be bool

In [ ]:
submission.to_csv('/kaggle/working/submission.csv', index=False)

## Final Summary

### Model Performance

| Model | CV Accuracy | Val Accuracy | AUC-ROC | F1 |
|-------|-------------|--------------|---------|-----|
| Logistic Regression | 79.26% | 79.41% | 0.8879 | 0.7968 |
| Random Forest | 79.78% | 80.56% | 0.8950 | 0.8021 |
| XGBoost (default) | 80.40% | 81.37% | 0.9114 | 0.8144 |
| XGBoost (tuned) | 80.41% | 81.25% | 0.9128 | 0.8160 |

**Best model:** XGBoost (tuned) — highest AUC-ROC (0.9128) and F1 (0.8160)
despite marginally lower val accuracy than XGBoost default. Used for
Kaggle submission.

### Business Hypothesis Results

| Hypothesis | Chi-squared | p-value | Result |
|---|---|---|---|
| HomePlanet affects transport rate | 324.90 | <0.0001 | Significant ✓ |
| Cabin Side affects transport rate | 91.06 | <0.0001 | Significant ✓ |
| Group travel affects transport rate | 112.10 | <0.0001 | Significant ✓ |

All three patterns identified during EDA are statistically confirmed —
none are sampling noise. HomePlanet shows the strongest effect
(chi-squared=324.90), consistent with its SHAP ranking as the second
most important feature.

### Key Findings

- **Top 3 SHAP features:** LeisureSpend, HomePlanet_Europa, CryoSleep.
  LeisureSpend ranked above CryoSleep because it is continuous (wider
  SHAP value range) and partially encodes CryoSleep — passengers in
  cryosleep always have zero leisure spending. SHAP distributes credit
  between correlated features rather than assigning it all to one.

- **Bayesian tuning:** Marginal improvement in AUC-ROC (+0.0014) and
  F1 (+0.0016) over default XGBoost. The narrow optimization range
  (0.785–0.808 across 25 trials) confirmed the model is near its ceiling
  with the current feature set — better features would help more than
  more tuning iterations.

- **Most interesting hypothesis result:** Group travel (chi-squared=112.10)
  produced a stronger test statistic than Cabin Side (91.06) despite a
  smaller raw gap (11 points vs 10 points). Both are real — large sample
  sizes make even modest effects statistically significant.


### What I Would Do Next

- Engineer feature interactions: HomePlanet × CryoSleep, Cabin_Deck × Side
- Stack or blend Logistic Regression + XGBoost — models make different
  errors which stacking can exploit
- Run Bayesian optimization with n_iter=50 and optimize for AUC-ROC
  directly rather than accuracy
- Cross-validate the full pipeline including feature engineering steps
  to get a more honest performance estimate